# Step 4 Walkthrough: Rule-Based Event Detection

This notebook starts after Step 3. It uses the smoothed possession sequence plus ball movement to create a first rule-based event detector.

Step 4 is intentionally separate from Step 2/3: it is the baseline detector and evaluation layer, not part of the master join or possession smoothing logic.


## Notebook Setup

The next code cell loads project paths, Step 4 configuration, and the rule-based detector functions. It uses the same `config.yaml` settings as `main.py`.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault(
    'MPLCONFIGDIR',
    str(PROJECT_ROOT / '.matplotlib_cache'),
)

import matplotlib.pyplot as plt

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from driblab.config import CONFIG_PATH, POSSESSION_SEQUENCE_DATA_DIR
from driblab.config import RULE_BASED_DATA_DIR, load_project_config
from driblab.models.rule_based_detector import Step4Config, run_step4

project_config = load_project_config(CONFIG_PATH)
PROJECT_ROOT


## Step 4 Input

Step 4 reads the Step 3 smoothed possession sequence table.

The table still has one row per reliable tracking frame, but it now includes columns such as `smoothed_possession_change`, `possession_team_change`, and `possession_player_change`. These columns are the event skeleton used by the rules.


In [ ]:
possession_sequence_path = (
    POSSESSION_SEQUENCE_DATA_DIR / 'possession_sequence_table.parquet'
)
preview_cols = [
    'match_id',
    'frame_id',
    'period_id',
    'tracking_match_clock_seconds',
    'event_label',
    'smoothed_possession_change',
    'possession_team_change',
    'possession_player_change',
    'ball_speed_mps',
    'ball_x_attacking',
]

possession_preview = pd.read_parquet(
    possession_sequence_path,
    columns=preview_cols,
)

print(f'Input path: {possession_sequence_path}')
print(f'Rows: {len(possession_preview):,}')
print(f'Matches: {possession_preview["match_id"].astype(str).nunique():,}')
possession_preview.head(10)


## Run Rule-Based Detection

The next code cell runs the Step 4 detector with thresholds and label mappings from `config.yaml`.

Rule examples:

- same-team smoothed possession player change -> `pass`
- opponent team change with fast ball -> `interception`
- opponent team change with slow ball -> `tackle`
- fast ball toward attacking goal -> `shot`
- ball near a boundary -> `out` or `corner`


In [ ]:
step4_config = project_config['step4']
step4_result = run_step4(
    Step4Config(
        input_table=possession_sequence_path,
        output_dir=RULE_BASED_DATA_DIR,
        evaluation_split=step4_config['evaluation_split'],
        shot_min_speed_mps=float(step4_config['shot_min_speed_mps']),
        shot_min_attacking_x=float(step4_config['shot_min_attacking_x']),
        shot_min_dx_attacking=float(step4_config['shot_min_dx_attacking']),
        interception_min_ball_speed_mps=float(
            step4_config['interception_min_ball_speed_mps']
        ),
        boundary_margin=float(step4_config['boundary_margin']),
        corner_y_margin=float(step4_config['corner_y_margin']),
        rule_classes=tuple(step4_config['rule_classes']),
        label_groups={
            key: tuple(value)
            for key, value in step4_config['label_groups'].items()
        },
    )
)

predictions = step4_result['predictions']
metrics = step4_result['metrics']
confusion = step4_result['confusion']
rule_summary = step4_result['summary']

print(step4_result['outputs'])
rule_summary


## Review Per-Class Metrics

This table shows precision, recall, F1, and support by broad event class on the configured evaluation split.


In [ ]:
metrics.sort_values('event_class')


## Evaluation Visuals

The next three plots show the rule-based detector from three angles: per-class metrics, a normalized confusion matrix, and true-vs-predicted class volume on the selected evaluation split.

In [ ]:
evaluation_split = step4_config['evaluation_split']
evaluation_rows = predictions[
    predictions['data_split'].eq(evaluation_split)
].copy()
class_order = list(step4_config['rule_classes'])

metric_cols = ['precision', 'recall', 'f1']
metrics_plot = (
    metrics.set_index('event_class')
    .reindex(class_order)[metric_cols]
    .fillna(0.0)
)

fig, ax = plt.subplots(figsize=(11, 5))
metrics_plot.plot(kind='bar', ax=ax, width=0.8)
ax.set_title(f'Rule-based detector metrics by class ({evaluation_split})')
ax.set_xlabel('Event class')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.legend(title='Metric')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
confusion_matrix = pd.crosstab(
    evaluation_rows['true_event_class'],
    evaluation_rows['rule_event_class'],
)
confusion_matrix = confusion_matrix.reindex(
    index=class_order,
    columns=class_order,
    fill_value=0,
).fillna(0).astype(int)
row_totals = confusion_matrix.sum(axis=1).astype(float)
row_totals = row_totals.where(row_totals.ne(0))
confusion_share = (
    confusion_matrix.astype(float)
    .div(row_totals, axis=0)
    .fillna(0.0)
)

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(confusion_share, cmap='Blues', vmin=0, vmax=1)
ax.set_title(f'Normalized confusion matrix ({evaluation_split})')
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_xticks(range(len(class_order)))
ax.set_yticks(range(len(class_order)))
ax.set_xticklabels(class_order, rotation=35, ha='right')
ax.set_yticklabels(class_order)

for row_idx, true_class in enumerate(class_order):
    for col_idx, predicted_class in enumerate(class_order):
        count = confusion_matrix.loc[true_class, predicted_class]
        share = confusion_share.loc[true_class, predicted_class]
        if count > 0:
            text_color = 'white' if share >= 0.5 else 'black'
            ax.text(
                col_idx,
                row_idx,
                f'{share:.0%}\n{count:,}',
                ha='center',
                va='center',
                color=text_color,
                fontsize=8,
            )

fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label='Row share')
plt.tight_layout()
plt.show()


In [ ]:
class_volume = pd.DataFrame(
    {
        'true_rows': evaluation_rows['true_event_class'].value_counts(),
        'predicted_rows': evaluation_rows['rule_event_class'].value_counts(),
    }
).reindex(class_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(11, 5))
class_volume.plot(kind='bar', ax=ax, width=0.8)
ax.set_title(f'True vs predicted class volume ({evaluation_split})')
ax.set_xlabel('Event class')
ax.set_ylabel('Rows')
ax.set_yscale('symlog')
ax.legend(['True rows', 'Predicted rows'])
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

class_volume


## Compare True Labels Against Rule Labels

This grouped table is a compact confusion-style view. It shows the most common combinations of provider-derived true class and rule-predicted class.


In [ ]:
prediction_counts = (
    predictions[predictions['data_split'].eq(step4_config['evaluation_split'])]
    .groupby(['true_event_class', 'rule_event_class'])
    .size()
    .reset_index(name='rows')
    .sort_values('rows', ascending=False)
)

prediction_counts.head(25)


## Inspect Rule Predictions

This sample keeps only frames where a rule predicted an event. Use it to audit which rule fired and whether the prediction agrees with the provider event label.


In [ ]:
sample_cols = [
    'match_id',
    'data_split',
    'period_id',
    'frame_id',
    'tracking_match_clock_seconds',
    'event_label',
    'true_event_class',
    'rule_event_class',
    'rule_reason',
    'previous_smoothed_possession_team_id',
    'smoothed_possession_team_id',
    'previous_smoothed_possession_player_id',
    'smoothed_possession_player_id',
    'ball_speed_mps',
    'ball_x_attacking',
]

predictions.loc[
    predictions['rule_event_class'].ne('no event'),
    sample_cols,
].head(30)


## Takeaways

- Step 4 is a rule-based baseline, not the final ML model.
- Use validation matches while changing thresholds.
- Use test matches only after the rules are frozen.
- The rule output gives a floor for later ML models to beat.
